# Delaunay-Watershed for geometry reconstruction: cell aggregate example

In this notebook:
- We show how to construct a multimaterial mesh from segmentation masks of cells in an aggregate.
- We show how to easily visualize the mesh.

# 0) Install Delaunay-Watershed

We install the viewing option from Delaunay-Watershed, based on Polyscope, to visualize the mesh later.

In a terminal, in the environment this notebook runs in (not from the notebook):

```shell
pip install "delaunay-watershed[viewing]"
```

## 1) Compute multimaterial mesh with Delaunay Watershed

A mesh is created via a "Mesh Reconstruction Algorithm". Although many variations can be created, we provide a default Mesh Reconstruction Algorithm:

In [1]:
from dw3d import get_default_mesh_reconstruction_algorithm

dist = 3
mesh_reconstruction_algorithm = get_default_mesh_reconstruction_algorithm(min_distance=dist, print_info=True)

This algorithm is able to reconstruct a 3D Mesh from a segmentation mask.

In [2]:
# Read segmentation image
import skimage.io as io

image_path = "../data/Images/1.tif"
image = io.imread(image_path)

# Use algorithm to create a mesh.
points, triangles, labels = mesh_reconstruction_algorithm.construct_mesh_from_segmentation_mask(image)
print("Done.\n---")

# The multimaterial mesh is:
# points: an array of 3D points (geometry)
# triangles: an array of triangles (topology)
# labels: an array of materials on each side of a triangle.
print(f"Created mesh has {len(points)} points and {len(triangles)} triangles.")

print(f"{points=}")
print(f"{triangles=}")
print(f"{labels=}")

Computing EDT ...


EDT computed in  0.3
Searching local extremas ...
Number of local minimas : 4804


Number of local maxes : 153
Delaunay Tesselation built in 0.15 seconds


Graph build in  0.239
Watershed done in 0.101 seconds.


Junction vertices relocated in 0.33 seconds: 46 of 48 moves accepted, 0 refused by the self-intersection guard
Done.
---
Created mesh has 1327 points and 2696 triangles.
points=array([[105.09161293,  98.36037792, 132.14635198],
       [ 98.05781367,  97.83901343, 135.10265704],
       [131.14393258,  97.5       , 107.06370787],
       ...,
       [ 94.        , 105.        ,  88.        ],
       [ 86.        , 104.        ,  82.        ],
       [ 79.        , 104.        ,  89.        ]], shape=(1327, 3))
triangles=array([[   0,  342,    1],
       [   0,    1, 1092],
       [   0,    1, 1273],
       ...,
       [1317, 1325, 1324],
       [1318, 1320, 1319],
       [1319, 1320, 1322]], shape=(2696, 3), dtype=uint64)
labels=array([[0, 2],
       [0, 1],
       [1, 2],
       ...,
       [1, 2],
       [1, 2],
       [1, 2]], shape=(2696, 2), dtype=uint64)


By default, the mesh has the same dimension as the image in pixels. The dimensions of the mesh can be set easily to match the real sample size:

The points are in the image's `(z, y, x)` order, so column 0 is scaled by the z size and columns 1 and 2 by the xy size (since 0.7.0; before, z and x were swapped). For a mask whose voxels are longer along z, prefer `construct_mesh_in_physical_units(mask, spacing)`, which resamples to cubic voxels first and is more accurate.

`data/Images/1.tif` is a 157 × 200 × 157 label stack whose file records no voxel size, so its true voxel size is not known: the sizes below only show the call. With your own data, give the sizes your microscope recorded.

In [3]:
from dw3d import center_points_around_zero, set_points_pixel_size

# data/Images/1.tif records no voxel size: these two sizes are illustrative, not the stack's own.
# Use your microscope's: the xy pixel size and the z step, in one unit (here micrometres).
known_xy_size_of_pixel = 0.1
known_z_size_of_pixel = 1.5

points = set_points_pixel_size(points, xy_pixel_size=known_xy_size_of_pixel, z_pixel_size=known_z_size_of_pixel)
# Now, the mesh has the real size of the sample, here it is between 0 and some size in micrometers.
# It's a simple scaling of dimensions, nothing more.

print(f"{points=}")

points=array([[157.63741939,   9.83603779,  13.2146352 ],
       [147.08672051,   9.78390134,  13.5102657 ],
       [196.71589888,   9.75      ,  10.70637079],
       ...,
       [141.        ,  10.5       ,   8.8       ],
       [129.        ,  10.4       ,   8.2       ],
       [118.5       ,  10.4       ,   8.9       ]], shape=(1327, 3))


Other operations are available, such as centering the points of the mesh around zero:

In [4]:
points = center_points_around_zero(points)
# Now the mesh is centered around 0
print(f"{points=}")

points=array([[ 4.13874194e+01, -6.39622076e-02,  5.41463520e+00],
       [ 3.08367205e+01, -1.16098657e-01,  5.71026570e+00],
       [ 8.04658989e+01, -1.50000000e-01,  2.90637079e+00],
       ...,
       [ 2.47500000e+01,  6.00000000e-01,  1.00000000e+00],
       [ 1.27500000e+01,  5.00000000e-01,  4.00000000e-01],
       [ 2.25000000e+00,  5.00000000e-01,  1.10000000e+00]],
      shape=(1327, 3))


Alternatively, you can set a global minimum and maximum for the size of the mesh:

In [5]:
from dw3d import set_points_global_min_max

points = set_points_global_min_max(points, global_min=-1, global_max=1)
# Now the mesh is between -1 and 1.
print(f"{points=}")

points=array([[ 4.27776945e-01, -6.61108088e-04,  5.59652217e-02],
       [ 3.18725793e-01, -1.19998612e-03,  5.90208341e-02],
       [ 8.31688877e-01, -1.55038760e-03,  3.00400081e-02],
       ...,
       [ 2.55813953e-01,  6.20155039e-03,  1.03359173e-02],
       [ 1.31782946e-01,  5.16795866e-03,  4.13436693e-03],
       [ 2.32558140e-02,  5.16795866e-03,  1.13695090e-02]],
      shape=(1327, 3))


The mesh can be saved easily for future use and analysis (with foambryo for example):

In [6]:
# The multimaterial mesh can be saved in a .vtk or .rec format. In binary or ASCII.
from dw3d import save_mesh_to_rec_mesh, save_mesh_to_vtk_mesh

save_mesh_to_rec_mesh(points, triangles, labels, "mesh_1.rec", binary_mode=False)
save_mesh_to_vtk_mesh(points, triangles, labels, "mesh_1.vtk", binary_mode=True)
print("Mesh saved.")

Mesh saved.


In general, the last constructed mesh is saved within the algorithm and the post-process can also be done with it:

In [7]:
# Alternatively, with last constructed mesh:
mesh_reconstruction_algorithm.construct_mesh_from_segmentation_mask(image)

mesh_reconstruction_algorithm.set_pixel_size(1, 1)
mesh_reconstruction_algorithm.center_around_zero()
mesh_reconstruction_algorithm.set_global_min_max(global_min=-1, global_max=1)


mesh_reconstruction_algorithm.save_to_rec_mesh("mesh_1.rec", binary_mode=False)
mesh_reconstruction_algorithm.save_to_vtk_mesh("mesh_1.vtk", binary_mode=True)

# The last mesh (with post-process operations) can be obtained:
post_processed_points, triangles, labels = mesh_reconstruction_algorithm.last_constructed_mesh

Computing EDT ...


EDT computed in  0.28
Searching local extremas ...
Number of local minimas : 4804


Number of local maxes : 153
Delaunay Tesselation built in 0.15 seconds


Graph build in  0.222
Watershed done in 0.131 seconds.


Junction vertices relocated in 0.3 seconds: 46 of 48 moves accepted, 0 refused by the self-intersection guard


## 2) Visualize mesh with our Polyscope viewer.

Similarly, the last mesh can be visualized, with our custom viewer:

In [8]:
from dw3d.viewing import plot_cells_polyscope

plot_cells_polyscope(mesh_reconstruction_algorithm)

# sometimes, it's better to scatter cells to understand the geometry
# plot_cells_polyscope(mesh_reconstruction_algorithm, view="Scattered", scattering_coeff=0.5)

[polyscope] Backend: openGL3_glfw -- Loaded openGL version: 4.1 Metal - 91.7
